# Workshop 3 · 03 — CV-Teaser: Wagon-Schaden → Werkstatt (Aufgabe)

**Ziel:** Der unstrukturierteste Datentyp — ein **Bild**. Eine **multimodale** AI Function bekommt den Foto-Pfad und liefert pro Wagon einen strukturierten Schadensbericht (Typ, Schweregrad, Empfehlung, UIC per OCR). Daraus leitest du den **Werkstatt-Auftrag** ab.

**Ausgangslage:** Wagon-Fotos in `Files/wagon_photos_w3` (aus Notebook 00).

**Deine Aufgaben:**
1. Die Fotos als Pfad-Spalte laden (Zelle unten ist bereitgestellt — der OneLake-`abfss://`-Pfad ist der Stolperstein).
2. Mit `ai.generate_response` (JSON-Schema, `col_types={'file_path': 'path'}`) pro Foto einen Schadensbericht erzeugen.
3. JSON parsen, Werkstatt-Zuordnung + `dispatch`-Flag ableiten und als Tabelle `schaden_work_orders` speichern.

Referenz: https://learn.microsoft.com/fabric/data-science/ai-functions/multimodal-overview

## Schritt 1 — Fotos laden (bereitgestellt)
Die AI Function braucht einen voll qualifizierten OneLake-`abfss://`-Pfad (kein relativer Pfad). Führe die Zelle aus.

In [ ]:
import synapse.ml.spark.aifunc as aifunc

# OneLake-ABFSS-Pfad zum Lakehouse bauen
onelake_endpoint = notebookutils.conf.get('trident.onelake.endpoint').replace('https://', '')
workspace_id = notebookutils.runtime.context['currentWorkspaceId']
lakehouse_id = notebookutils.runtime.context['defaultLakehouseId']

base_path = f'abfss://{workspace_id}@{onelake_endpoint}/{lakehouse_id}/Files/wagon_photos_w3'

files_df = aifunc.list_file_paths(base_path)
display(files_df)

## Schritt 2 — Schadensbericht mit multimodaler AI Function

**Aufgabe:**
1. Definiere ein JSON-Schema mit `damage_type` (enum: rost/kratzer/delle/graffiti/riss/kein_schaden), `severity` (integer 1–5), `recommended_action` (string), `uic_number` (string), `safety_relevant` (boolean).
2. Rufe `files_df.ai.generate_response(prompt=..., col_types={'file_path': 'path'}, response_format=..., output_col='report_json')` mit einem Prompt als Prüfingenieur auf.

In [ ]:
# This code uses AI. Always review output for mistakes.

# TODO: JSON-Schema fuer den Schadensbericht definieren
report_schema = {
    # 'type': 'json_schema',
    # 'json_schema': { 'name': 'schadensbericht', 'strict': True, 'schema': { ... } }
}

# TODO: Prompt als Pruefingenieur formulieren
# prompt = 'Du bist Pruefingenieur fuer Schienenfahrzeuge. Analysiere das Foto ...'

# TODO: multimodale AI Function pro Foto aufrufen (col_types markiert file_path als Bildpfad)
# reports = files_df.ai.generate_response(
#     prompt=prompt,
#     col_types={'file_path': 'path'},
#     response_format=report_schema,
#     output_col='report_json',
# )
# display(reports.select('file_path', 'report_json'))

## Schritt 3 — Werkstatt-Auftrag ableiten und speichern

**Aufgabe:** Parse `report_json` mit `from_json`, leite eine `workshop`-Spalte ab (rost/kratzer → Karosserie/Lack, graffiti → Reinigung, delle/riss → Strukturpruefung) und ein `dispatch`-Flag (`severity >= 4` **oder** `safety_relevant`). Speichere als Tabelle `schaden_work_orders`.

In [ ]:
from pyspark.sql.functions import from_json, col, when, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType

# TODO: Struktur-Schema passend zum report_json definieren
# rschema = StructType([...])

# TODO: JSON parsen und Felder selektieren
# findings = (reports
#             .withColumn('r', from_json(col('report_json'), rschema))
#             .select(col('file_path').cast('string').alias('image_path'),
#                     'r.damage_type', 'r.severity', 'r.recommended_action',
#                     'r.uic_number', 'r.safety_relevant'))

# TODO: workshop-Zuordnung + dispatch-Flag ableiten
# work = (findings
#         .withColumn('workshop', when(...).otherwise(lit('-')))
#         .withColumn('dispatch', (col('severity') >= 4) | (col('safety_relevant') == True))
#         .withColumn('created_at', current_timestamp()))

# TODO: als Tabelle schaden_work_orders speichern
# work.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('schaden_work_orders')
# display(work.orderBy(col('severity').desc()))

### ✅ Ergebnis-Check & Operationalisierung
Aus Bildern wurde die strukturierte Tabelle `schaden_work_orders` mit Werkstatt-Zuordnung und `dispatch`-Flag. Im echten Workload folgt:
- **Activator**-Regel auf `dispatch = true` → Teams-Nachricht an die Werkstatt-Disposition.
- **Power BI** (Direct Lake) Dashboard über offene Aufträge.